Importation des bibliotheque et vérification du fichier

In [6]:
import pandas as pd
import gzip
import csv
from pathlib import Path

DATA_PATH = Path("../data/raw/full-2025.csv.gz")

print("fichier trouvé: ", DATA_PATH.exists())

if DATA_PATH.exists():
    print("Taille :", round(DATA_PATH.stat().st_size / 1024**2, 2), "Mo"
    )

    with gzip.open(DATA_PATH, "rt", encoding="utf-8-sig") as f:
        premiere_ligne = f.readline()

    separateur = ";" if premiere_ligne.count(";") > premiere_ligne.count(",") else ","

    print("Séparateur détecté :", repr(separateur))
    print("En-tete :", next(csv.reader([premiere_ligne], delimiter=separateur)))

fichier trouvé:  True
Taille : 110.88 Mo
Séparateur détecté : ','
En-tete : ['id_mutation', 'date_mutation', 'numero_disposition', 'nature_mutation', 'valeur_fonciere', 'adresse_numero', 'adresse_suffixe', 'adresse_nom_voie', 'adresse_code_voie', 'code_postal', 'code_commune', 'nom_commune', 'code_departement', 'ancien_code_commune', 'ancien_nom_commune', 'id_parcelle', 'ancien_id_parcelle', 'numero_volume', 'lot1_numero', 'lot1_surface_carrez', 'lot2_numero', 'lot2_surface_carrez', 'lot3_numero', 'lot3_surface_carrez', 'lot4_numero', 'lot4_surface_carrez', 'lot5_numero', 'lot5_surface_carrez', 'nombre_lots', 'code_type_local', 'type_local', 'surface_reelle_bati', 'nombre_pieces_principales', 'code_nature_culture', 'nature_culture', 'code_nature_culture_speciale', 'nature_culture_speciale', 'surface_terrain', 'longitude', 'latitude']


Lecture d'un echantillon

In [7]:
df = pd.read_csv(
    DATA_PATH,
    compression="gzip",
    sep=separateur,
    nrows=5000,
    dtype=str,
    encoding="utf-8-sig"
    )

print("nb ligne", len(df))
print("nb colonne", len(df.columns))

display(df.head(10))


nb ligne 5000
nb colonne 40


,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
0,2025-1,2025-01-07,000001,Vente,468000.00,NaN,NaN,FARGES,B078,01550,...,NaN,NaN,NaN,J,jardins,NaN,NaN,78,5.907192,46.17086538221288
1,2025-1,2025-01-07,000001,Vente,468000.00,454,NaN,RUE DE LA REPUBLIQUE,0090,01550,...,Dépendance,0,0,S,sols,NaN,NaN,133,5.907199399999999,46.17077450112271
2,2025-1,2025-01-07,000001,Vente,468000.00,454,NaN,RUE DE LA REPUBLIQUE,0090,01550,...,Maison,111,5,S,sols,NaN,NaN,133,5.907199399999999,46.17077450112271
3,2025-2,2025-01-06,000001,Vente,180000.00,NaN,NaN,LE VILLAGE,B158,01200,...,NaN,NaN,NaN,S,sols,NaN,NaN,46,5.801625249999999,46.16456865016352
4,2025-2,2025-01-06,000001,Vente,180000.00,NaN,NaN,LE VILLAGE,B158,01200,...,NaN,NaN,NaN,J,jardins,NaN,NaN,17,5.801687649999999,46.164608110901085
5,2025-2,2025-01-06,000001,Vente,180000.00,66,NaN,RUE DE LA FRUITIERE,0050,01200,...,Appartement,87,4,S,sols,NaN,NaN,76,5.801489799999999,46.16454464920855
6,2025-3,2025-01-06,000001,Vente,490000.00,NaN,NaN,PETIT CORGENON,B060,01310,...,NaN,NaN,NaN,S,sols,NaN,NaN,415,5.1537907999999995,46.20595733801825
7,2025-3,2025-01-06,000001,Vente,490000.00,459,B,CHE DU PETIT CORGENON,0356,01310,...,Dépendance,0,0,AG,terrains d'agrément,NaN,NaN,2452,5.15504275,46.20618750301992
8,2025-3,2025-01-06,000001,Vente,490000.00,459,B,CHE DU PETIT CORGENON,0356,01310,...,Dépendance,0,0,AG,terrains d'agrément,NaN,NaN,2452,5.15504275,46.20618750301992
9,2025-3,2025-01-06,000001,Vente,490000.00,459,B,CHE DU PETIT CORGENON,0356,01310,...,Dépendance,0,0,AG,terrains d'agrément,NaN,NaN,2452,5.15504275,46.20618750301992


Diagnostic qualité

In [10]:
diagnostic = pd.DataFrame({
    "Colonne": df.columns,
    "Type": df.dtypes.astype(str).values,
    "Valeur manquantes": df.isna().sum().values,
    "% manquant" :(
        df.isna().mean().values * 100
    ).round(2),
    "Valeur distinctes" : df.nunique(dropna=True).values
})

display(diagnostic)

,Colonne,Type,Valeur manquantes,% manquant,Valeur distinctes
0,id_mutation,str,0,0.00,1771
1,date_mutation,str,0,0.00,55
2,numero_disposition,str,0,0.00,6
3,nature_mutation,str,0,0.00,4
4,valeur_fonciere,str,30,0.60,850
5,adresse_numero,str,2514,50.28,470
6,adresse_suffixe,str,4839,96.78,10
7,adresse_nom_voie,str,29,0.58,1792
8,adresse_code_voie,str,29,0.58,697
9,code_postal,str,29,0.58,59


Qualité des données

In [12]:
qualite = pd.DataFrame({
    "Colonne": df.columns,
    "Valeur manquantes": df.isna().sum().values,
    "% manquant" :(
        df.isna().mean().values * 100
    ).round(2),
    "Valeur uniques": df.nunique(dropna=True).values
})

display(
    qualite.sort_values("% manquant",ascending=False)
)

,Colonne,Valeur manquantes,% manquant,Valeur uniques
13,ancien_code_commune,5000,100.00,0
27,lot5_surface_carrez,5000,100.00,0
14,ancien_nom_commune,5000,100.00,0
26,lot5_numero,4999,99.98,1
25,lot4_surface_carrez,4996,99.92,1
17,numero_volume,4990,99.80,6
23,lot3_surface_carrez,4987,99.74,5
24,lot4_numero,4980,99.60,7
16,ancien_id_parcelle,4954,99.08,30
22,lot3_numero,4939,98.78,21


Repartition des biens 

In [14]:
display(
    df["type_local"]
    .fillna("Non renseigné")
    .value_counts()
    .to_frame("Nombre de lignes")
)

,Nombre de lignes
type_local,
Non renseigné,2434
Dépendance,1185
Maison,697
Appartement,533
Local industriel. commercial ou assimilé,151


Repar des natures mutation

In [15]:
display(
    df["nature_mutation"]
    .fillna("Non renseigné")
    .value_counts()
    .to_frame("Nombre de lignes")
)

,Nombre de lignes
nature_mutation,
Vente,4667
Vente en l'état futur d'achèvement,261
Echange,63
Vente terrain à bâtir,9


Verif mutation répétées

In [17]:
cles = ["id_mutation", "numero_disposition"]

nb_lignes = len(df)

nb_transactions = (
    df[cles]
    .drop_duplicates()
    .shape[0]
)

print("nb ligne: ", nb_lignes)
print("nb mutation :", nb_transactions)

print("nb moyen par transac :", round(nb_lignes / nb_transactions, 2) 
)

nb ligne:  5000
nb mutation : 1779
nb moyen par transac : 2.81


Analyse type de bien

In [18]:
types_biens = (
    df["type_local"]
    .fillna("Non renseigné")
    .value_counts()
    .reset_index()
)

types_biens.columns = ["Type de bien", "Nombre de lignes"]

types_biens["Pourcentage"]= (
    types_biens["Nombre de lignes"] / len(df) * 100
).round(2)

display(types_biens)

,Type de bien,Nombre de lignes,Pourcentage
0,Non renseigné,2434,48.68
1,Dépendance,1185,23.70
2,Maison,697,13.94
3,Appartement,533,10.66
4,Local industriel. commercial ou assimilé,151,3.02


Analyse répétition mutation

In [20]:
cles =["id_mutation", "numero_disposition"]

nb_transactions = df[cles].drop_duplicates().shape[0]

print("Nombre de lignes :", len(df))
print("Nombre de transactions :", nb_transactions)

print (
    "Nombre moyen de lignes par transaction :",
    round(len(df) / nb_transactions, 2)

)

display(
    df.groupby(cles)
    .size()
    .sort_values(ascending=False)
    .head(10)
    .to_frame("Nombre de lignes")
)

Nombre de lignes : 5000
Nombre de transactions : 1779
Nombre moyen de lignes par transaction : 2.81


,,Nombre de lignes
id_mutation,numero_disposition,
2025-610,000001,33
2025-813,000001,32
2025-903,000001,31
2025-422,000001,29
2025-661,000001,25
2025-404,000003,25
2025-92,000001,23
2025-1601,000001,21
2025-344,000001,21


Extraction données Rhône

In [ ]:
# from pathlib import Path
# import pandas as pd

# source = Path("../data/raw/full-2025.csv.gz")
# destination = Path("../data/processed/dvf_rhone_2025.csv")

# destination.parent.mkdir(parents=True, exist_ok=True)

# premier_bloc = True
# nombre_lignes = 0

# for bloc in pd.read_csv(
#     source,
#     compression="gzip",
#     sep=",",
#     dtype=str,
#     chunksize=100_00
# ):
#     rhone = bloc.loc[
#         bloc["code_departement"].eq("69")
#     ]

#     if rhone.empty:
#         continue

#     rhone.to_csv(
#         destination,
#         mode="w" if premier_bloc else "a",
#         header=premier_bloc,
#         index=False,
#         encoding="utf-8-sig"
#     )

In [32]:
df_rhone = pd.read_csv(
    "../data/processed/dvf_rhone_2025.csv",
    dtype=str,
    encoding="utf-8-sig"
)

print("Nombre de lignes :", len(df_rhone))

print("Departmeents présents :", df_rhone["code_departement"].unique()
)

print("nombre de communes :", 
    df_rhone["code_commune"].nunique()
)

display(
    df_rhone[["code_commune", "nom_commune"]]
    .drop_duplicates()
    .sort_values("nom_commune")
    .head(30)
)

Nombre de lignes : 8431
Departmeents présents : <StringArray>
['69']
Length: 1, dtype: str
nombre de communes : 33


,code_commune,nom_commune
20,69029,Bron
150,69270,Chaponnay
867,69271,Chassieu
580,69299,Colombier-Saugnieu
492,69272,Communay
334,69273,Corbas
30,69275,Decines-Charpieu
35,69276,Feyzin
19,69277,Genas
247,69279,Jonage


In [33]:
print("Première date :", df_rhone["date_mutation"].min())
print("Dernière date :", df_rhone["date_mutation"].max())

print("Première mutation :", df_rhone["id_mutation"].iloc[0])
print("Dernière mutation :", df_rhone["id_mutation"].iloc[-1])

Première date : 2025-03-20
Dernière date : 2025-12-31
Première mutation : 2025-943007
Dernière mutation : 2025-945724


In [34]:
source = "../data/raw/full-2025.csv.gz"

total_national = 0
total_rhone = 0 
communes = set()
dates_rhone = []

for bloc in pd.read_csv(
    source,
    compression="gzip",
    dtype=str,
    chunksize=100_000
):
    total_national += len(bloc)

    rhone = bloc[
        bloc["code_departement"].str.strip() == "69"
    ]

    total_rhone += len((rhone))

    communes.update(
        rhone["code_commune"].dropna().unique()

    )

    dates_rhone.extend(
        rhone["date_mutation"].dropna().tolist()
    )

print("ligne nationales :", total_national)
print("ligne rhone : ", total_rhone)
print("commune rhone :", len(communes))

if dates_rhone:
    print("premiere date :", min(dates_rhone))
    print("derniere date :" , max(dates_rhone))


ligne nationales : 3793113
ligne rhone :  85850
commune rhone : 274
premiere date : 2025-01-02
derniere date : 2025-12-31


In [36]:
source = Path("../data/raw/full-2025.csv.gz")
destination = Path('../data/processed/dvf_rhone_2025_complet.csv')

destination.parent.mkdir(parents=True, exist_ok=True)

total_rhone = 0
premier_bloc = True

for i, bloc in enumerate(
    pd.read_csv(
        source,
        compression="gzip",
        sep=",",
        dtype=str,
        chunksize=100_000
    ),
    start=1
):
    rhone=bloc.loc[
        bloc["code_departement"].str.strip().eq("69")

    ]
    if not rhone.empty:
        rhone.to_csv(
            destination,
            mode="w" if premier_bloc else "a",
            header=premier_bloc,
            index=False,
            encoding="utf-8-sig"
        )

        total_rhone += len(rhone)
        premier_bloc = False

    if i % 10 == 0:
        print(f"Bloc {i} terminé : {total_rhone} lignes Rhône")

print("Extraction terminée")
print("ligne exporté :", total_rhone)

df_rhone = pd.read_csv(
    destination,
    dtype=str,
    encoding="utf-8-sig"
)

Bloc 10 terminé : 0 lignes Rhône
Bloc 20 terminé : 0 lignes Rhône
Bloc 30 terminé : 85850 lignes Rhône
Extraction terminée
ligne exporté : 85850


In [37]:
df_rhone = pd.read_csv(
    "../data/processed/dvf_rhone_2025_complet.csv",
    dtype=str,
    encoding="utf-8-sig"
)

print("nb lignes :", len(df_rhone))
print("nb commune :", df_rhone["code_commune"].nunique())
print("departement :", df_rhone["code_departement"].unique())
print("premiere date :", df_rhone["date_mutation"].min())
print("derniere date :", df_rhone["date_mutation"].max())

assert len(df_rhone) == 85850
assert df_rhone["code_commune"].nunique() ==274

print("extract validé")

nb lignes : 85850
nb commune : 274
departement : <StringArray>
['69']
Length: 1, dtype: str
premiere date : 2025-01-02
derniere date : 2025-12-31
extract validé
